# v6: select unfolded DL by its validation gap to GS

P=10 fine-tunes the v3 model; P=30 fine-tunes the v5 SNR-conditioned model. Each is repeated with three shuffle seeds. If Colab denies a GPU, run the GS validation-cache cell on CPU and stop there. When a T4 becomes available, rerun the setup cells and start training. GS estimates are never fed to the DL model. Completed GS curves and training checkpoints are saved in Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')
import os, torch
root = '/content/gdrive/MyDrive'
required = [
    f'{root}/DL_dataset_RSR25dB/{split}_P{p}.mat'
    for p in (10, 30) for split in ('train', 'validation', 'test')
] + [
    f'{root}/unfolded_dl_v3_highsnr_results/model_P10_unfolded_v2.pt',
    f'{root}/unfolded_dl_v5_snr_aware_results/model_P30_snr_conditioned_only.pt',
]
missing = [path for path in required if not os.path.isfile(path)]
assert not missing, f'Missing Drive files: {missing}'
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
print('Dataset and pretrained models found.')

In [ ]:
from google.colab import files
uploaded = files.upload()
required_scripts = {
    'colab_unfolded_dl_v3_highsnr.py',
    'colab_unfolded_dl_v5_snr_aware.py',
    'colab_unfolded_dl_v6_gs_gap.py',
}
missing = required_scripts.difference(uploaded)
assert not missing, f'Upload these scripts: {sorted(missing)}'
print('All three scripts uploaded.')

## CPU step: cache validation GS curves

Run this once with a CPU runtime if GPU access is unavailable. It stores GS validation results in Drive. You can also run it on T4; the training script will reuse the cache.

In [ ]:
import sys
sys.path.insert(0, '/content')
import colab_unfolded_dl_v6_gs_gap as v6
os.makedirs(v6.OUTPUT_ROOT, exist_ok=True)
for p in (10, 30):
    path = os.path.join(v6.DATA_ROOT, f'validation_P{p}.mat')
    result = v6.cached_gs(p, 'validation', path)
    print(f'P={p} GS validation NMSE by SNR:', result['nmse_db_by_snr'])

## Training: run when T4 is available

In [ ]:
!python /content/colab_unfolded_dl_v6_gs_gap.py

In [ ]:
import json
result_dir = '/content/gdrive/MyDrive/unfolded_dl_v6_gs_gap_results'
with open(f'{result_dir}/summary_unfolded_v6_gs_gap.json') as f:
    results = json.load(f)
for result in results:
    print(f"P={result['P']} seed={result['seed']} selected epoch={result['best_epoch']}")
    for snr, gap in result['test_dl_minus_gs_db_by_snr'].items():
        print(f'  {snr:>3} dB: DL minus GS = {gap:+.3f} dB')